# MuSeg-AI Thigh — Augmented Dataset Evaluation — GPU Lambda

GPU-accelerated evaluation of MuSeg-AI Thigh segmentations on the 20-volume
augmented dataset (`our_augmented_dataset/`), against the myosegmenTUM-derived
ground truth (Gracilis / Hamstrings / Quadriceps / Sartorius, bilateral).

- **Predictions**: `~/museg_augmented_segs/`
- **GT**: `~/our_augmented_dataset/{stem}_seg.nii.gz` (labels 1-8: L/R x
  Gracilis/Hamstrings/Quadriceps/Sartorius)

## Upload to Lambda
```bash
rsync -avz --mkpath -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/our_augmented_dataset/ \
  ubuntu@<IP>:~/our_augmented_dataset/

rsync -avz --mkpath -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/museg/augmented_segs/ \
  ubuntu@<IP>:~/museg_augmented_segs/
```

## Download results
```bash
rsync -avz --mkpath -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  ubuntu@<IP>:~/museg_augmented_results/ \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/museg/codes/results_augmented/
```

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'SimpleITK', 'pandas', 'numpy<2'])
print('Dependencies ready.')

In [ ]:
import glob, os, re
import numpy as np
import pandas as pd
import SimpleITK as sitk
import torch
import torch.nn.functional as F

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  {torch.cuda.get_device_name(0)}')
    free, total = torch.cuda.mem_get_info(0)
    print(f'  VRAM: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total')

In [ ]:
def _to_bool(arr, device):
    return torch.from_numpy(arr.astype(np.uint8)).bool().to(device)

def _erode3d(mask):
    inv = (~mask).float().view(1, 1, *mask.shape)
    inv_pad = F.pad(inv, [1, 1, 1, 1, 1, 1], value=0)
    return ~(F.max_pool3d(inv_pad, kernel_size=3, stride=1, padding=0).squeeze(0).squeeze(0) > 0)

def _dilate3d(mask, dist=1):
    m = mask.float().view(1, 1, *mask.shape)
    for _ in range(dist):
        m = F.max_pool3d(F.pad(m, [1,1,1,1,1,1], value=0), kernel_size=3, stride=1, padding=0)
    return m.squeeze(0).squeeze(0) > 0

def _surface3d(mask):
    return mask & ~_erode3d(mask)

def _surface_pts(mask, spacing):
    surf = _surface3d(mask)
    if surf.sum() == 0: return None
    idx = surf.nonzero(as_tuple=False).float()
    scale = torch.tensor([spacing[2], spacing[1], spacing[0]], dtype=torch.float32, device=mask.device)
    return idx * scale

def hausdorff_gpu(pred, gt, spacing, chunk_a=512, chunk_b=512):
    # Both sides chunked — a single un-chunked side blows up memory to
    # O(chunk_a * len(b)) when a mask is noisy/degenerate (e.g. near-volume-
    # wide false positives turn most voxels into "surface" voxels).
    if pred.sum() == 0 or gt.sum() == 0: return float('nan')
    pp, gp = _surface_pts(pred, spacing), _surface_pts(gt, spacing)
    if pp is None or gp is None: return float('nan')
    def directed_max(a, b):
        max_min = torch.tensor(0.0, device=a.device)
        for i in range(0, len(a), chunk_a):
            a_chunk   = a[i:i + chunk_a]
            min_dists = torch.full((len(a_chunk),), float('inf'), device=a.device)
            for j in range(0, len(b), chunk_b):
                d         = torch.cdist(a_chunk, b[j:j + chunk_b])
                min_dists = torch.minimum(min_dists, d.min(dim=1).values)
            max_min = torch.maximum(max_min, min_dists.max())
        return max_min.item()
    return max(directed_max(pp, gp), directed_max(gp, pp))

def dice_gpu(pred, gt):
    return (2*(pred&gt).float().sum() / (pred.float().sum()+gt.float().sum()+1e-8)).item()

def jaccard_gpu(pred, gt):
    return ((pred&gt).float().sum() / ((pred|gt).float().sum()+1e-8)).item()

def volume_similarity_gpu(pred, gt):
    ps, gs = pred.float().sum(), gt.float().sum()
    return (1-(ps-gs).abs()/(ps+gs+1e-8)).item()

def false_negative_gpu(pred, gt):
    return ((~pred&gt).float().sum()/(gt.float().sum()+1e-8)).item()

def false_positive_gpu(pred, gt):
    return ((pred&~gt).float().sum()/(pred.float().sum()+1e-8)).item()

def bce_gpu(pred, gt, eps=1e-7):
    pf = pred.float().clamp(eps,1-eps); gf = gt.float()
    return (-gf*pf.log()-(1-gf)*(1-pf).log()).mean().item()

def boundary_iou_3d_gpu(pred, gt, dist=1):
    bp = _dilate3d(_surface3d(pred), dist); bg = _dilate3d(_surface3d(gt), dist)
    return ((bp&bg).float().sum()/((bp|bg).float().sum()+1e-8)).item()

def inter_slice_dice_gpu(pred):
    if pred.shape[0] < 2: return float('nan')
    a, b = pred[:-1].float(), pred[1:].float()
    inter = (a*b).sum(dim=(1,2)); denom = a.sum(dim=(1,2))+b.sum(dim=(1,2))
    valid = denom > 0
    return (2*inter[valid]/denom[valid]).mean().item() if valid.any() else 0.0

print('GPU helpers ready.')

In [ ]:
# our_augmented_dataset combined_gt label convention (from create_augmented_dataset.ipynb):
#   1=L_Gracilis 2=L_Hamstrings 3=L_Quadriceps 4=L_Sartorius
#   5=R_Gracilis 6=R_Hamstrings 7=R_Quadriceps 8=R_Sartorius
# Hamstrings = biceps femoris (long+short) + semitendinosus + semimembranosus (compound).
# Quadriceps = vastus lateralis/medialis/intermedius + rectus femoris (compound).
# Each MUSCLES entry below ORs together the L+R GT labels for one group, so
# algorithms that don't distinguish side (single bilateral label) compare cleanly.

def read_gt(seg_path, group_labels):
    arr = sitk.GetArrayFromImage(sitk.ReadImage(seg_path)).astype(np.int32)
    return np.isin(arr, group_labels).astype(np.uint8)

def get_spacing(seg_path):
    return sitk.ReadImage(seg_path).GetSpacing()

print('GT helpers ready.')

In [ ]:
BOUNDARY_DISTANCE = 1
DATA_DIR   = os.path.expanduser('~/our_augmented_dataset')
SEG_DIR    = os.path.expanduser('~/museg_augmented_segs')
RESULT_DIR = os.path.expanduser('~/museg_augmented_results')
ALGO_TAG   = 'museg'
os.makedirs(RESULT_DIR, exist_ok=True)

# (muscle_name, gt_labels [L,R combined], pred_spec)
MUSCLES = [
    ('gracilis', [1, 5], [6]),
    ('sartorius', [4, 8], [5]),
    ('hamstrings', [2, 6], [7, 8, 9, 10]),
    ('quadriceps', [3, 7], [1, 2, 3, 4]),
]

water_files = sorted(glob.glob(os.path.join(DATA_DIR, '*_augmented*_water.nii.gz')))
print(f'DATA dir: {DATA_DIR}')
print(f'SEG dir : {SEG_DIR}')
print(f'Found   : {len(water_files)} water NIfTI volumes')

In [ ]:
def load_pred(pred_path, pred_labels, gt_shape, gt_spacing):
    if not os.path.exists(pred_path):
        return None
    pred_sitk = sitk.ReadImage(pred_path)
    seg_raw   = sitk.GetArrayFromImage(pred_sitk)
    if seg_raw.shape != gt_shape:
        ref = sitk.GetImageFromArray(np.zeros(gt_shape, dtype=np.int32))
        ref.SetSpacing(gt_spacing)
        pred_sitk = sitk.Resample(pred_sitk, ref, sitk.Transform(), sitk.sitkNearestNeighbor, 0)
        seg_raw   = sitk.GetArrayFromImage(pred_sitk)
    pred_np = np.zeros(gt_shape, dtype=np.uint8)
    for lbl in pred_labels:
        pred_np |= (seg_raw == lbl).astype(np.uint8)
    return pred_np


def evaluate_muscle(muscle_name, gt_labels, pred_labels):
    results = []
    for water_path in water_files:
        base_stem = os.path.basename(water_path).replace('_water.nii.gz', '')
        gt_path   = os.path.join(DATA_DIR, f'{base_stem}_seg.nii.gz')
        if not os.path.exists(gt_path):
            print(f'  [skip] GT missing: {base_stem}'); continue
        gt_bin  = read_gt(gt_path, gt_labels)
        spacing = get_spacing(gt_path)

        pred_path = os.path.join(SEG_DIR, base_stem + '_dseg.nii.gz')
        pred_np = load_pred(pred_path, pred_labels, gt_bin.shape, spacing)
        if pred_np is None:
            print(f'  [skip] prediction missing: {base_stem}'); continue

        pred_t = _to_bool(pred_np, DEVICE)
        gt_t   = _to_bool(gt_bin,  DEVICE)
        with torch.no_grad():
            row = {
                'sample':                               base_stem,
                f'{muscle_name}_dice':                  dice_gpu(pred_t, gt_t),
                f'{muscle_name}_hausdorff':             hausdorff_gpu(pred_t, gt_t, spacing),
                f'{muscle_name}_jaccard':               jaccard_gpu(pred_t, gt_t),
                f'{muscle_name}_volume_similarity':     volume_similarity_gpu(pred_t, gt_t),
                f'{muscle_name}_false_negative':        false_negative_gpu(pred_t, gt_t),
                f'{muscle_name}_false_positive':        false_positive_gpu(pred_t, gt_t),
                f'{muscle_name}_bce':                   bce_gpu(pred_t, gt_t),
                f'{muscle_name}_boundary_iou_3d':       boundary_iou_3d_gpu(pred_t, gt_t, BOUNDARY_DISTANCE),
                f'{muscle_name}_inter_slice_dice_pred': inter_slice_dice_gpu(pred_t),
                f'{muscle_name}_inter_slice_dice_gt':   inter_slice_dice_gpu(gt_t),
            }
        results.append(row)
        dice_val = row[f'{muscle_name}_dice']; hd_val = row[f'{muscle_name}_hausdorff']
        print(f'  {base_stem:<32s}  dice={dice_val:.4f}  hd={hd_val:.2f}mm')
    df = pd.DataFrame(results)
    csv_path = os.path.join(RESULT_DIR, f'df_{muscle_name}_{ALGO_TAG}_augmented.csv')
    df.to_csv(csv_path, index=False)
    print(f'  Saved {len(df)} rows -> {csv_path}')
    return df


dfs = {}
for muscle_name, gt_labels, pred_spec in MUSCLES:
    print(f'\n-- {muscle_name}  (GT={gt_labels}) --')
    dfs[muscle_name] = evaluate_muscle(muscle_name, gt_labels, pred_spec)
print('\nDone.')

In [ ]:
summary_rows = []
for muscle_name, df in dfs.items():
    if df.empty: continue
    summary_rows.append({
        'muscle': muscle_name, 'n': len(df),
        'dice_mean':      df[f'{muscle_name}_dice'].mean(),
        'dice_std':       df[f'{muscle_name}_dice'].std(),
        'hausdorff_mean': df[f'{muscle_name}_hausdorff'].mean(),
        'hausdorff_std':  df[f'{muscle_name}_hausdorff'].std(),
    })
summary      = pd.DataFrame(summary_rows).set_index('muscle')
summary_path = os.path.join(RESULT_DIR, f'summary_{ALGO_TAG}_augmented.csv')
summary.to_csv(summary_path)
print(f'Summary saved -> {summary_path}\n')
print(summary.round(4).to_string())